In [0]:
from pyspark.sql.functions import col, lower, upper, initcap, month, year, when, count,regexp_replace
from pyspark.sql.types import DoubleType, IntegerType

In [0]:
df_bronze = spark.table("projeto_profissional_bigdata.datatran_2018.bronze_datatran2018")

In [0]:
#Conferência do esquema

df_bronze.printSchema()

In [0]:
# Visualizando os dados
df_bronze.limit(10).toPandas()

In [0]:
#Selecionando colunas

df_silver = df_bronze.select("id", "data_inversa","uf", "br", "km", "municipio", "causa_acidente", "tipo_acidente")

df_silver.limit(5).toPandas()

In [0]:
#Removendo linhas com valores nulos

df_silver = df_silver.dropna(
    subset = ["id","data_inversa","uf","br","km", "municipio","causa_acidente","tipo_acidente"]
)

df_silver.limit(5).toPandas()


In [0]:
#Renomeando coluna data_inversa para data_acidente
df_silver = df_silver.withColumnRenamed("data_inversa", "data_acidente")



In [0]:
#Transformando colunas Municipio, Causa_acidente e Tipo acidente em minusculas

df_silver = df_silver.withColumn("municipio", initcap(col("municipio")))
df_silver = df_silver.withColumn("causa_acidente", lower(col("causa_acidente")))
df_silver = df_silver.withColumn("tipo_acidente", lower(col("tipo_acidente")))

df_silver.limit(5).toPandas()


In [0]:
#criando coluna mes acidente

df_silver = df_silver.withColumn("mes_acidente", month(col("data_acidente")))
df_silver = df_silver.withColumn("ano_acidente", year(col("data_acidente")))

df_silver.limit(5).toPandas()


In [0]:
#Mudando os valores da coluna mes acidente para os nomes dos meses
df_silver = df_silver.withColumn(
"nome_mes", 
when(df_silver.mes_acidente == 1, "Janeiro")
.when(df_silver.mes_acidente == 2, "Fevereiro")
.when(df_silver.mes_acidente == 3, "Março")
.when(df_silver.mes_acidente == 4, "Abril")
.when(df_silver.mes_acidente == 5, "Maio")
.when(df_silver.mes_acidente == 6, "Junho")
.when(df_silver.mes_acidente == 7, "Julho")
.when(df_silver.mes_acidente == 8, "Agosto")
.when(df_silver.mes_acidente == 9, "Setembro")
.when(df_silver.mes_acidente == 10, "Outubro")
.when(df_silver.mes_acidente == 11, "Novembro")
.otherwise("Dezembro")
)

df_silver.limit(5).toPandas()


In [0]:
#Contando linhas na coluna br com valor NA

df_silver.filter(col("br") == "NA").count()

In [0]:
#Substituindo valores NA por None

df_silver = df_silver.withColumn("br", when(col("br") == "NA", None).otherwise(col("br")))

In [0]:
df_silver = df_silver.dropna(subset = ["br"])

In [0]:
#trocando o tipo das colunas BR e KM para double

df_silver = df_silver.withColumn("br", col("br").cast(IntegerType()))

df_silver.limit(5).toPandas()


In [0]:
#Trocando virgula por ponto na coluna KM

df_silver = df_silver.withColumn("km", regexp_replace(col("km"), ",", "."))
df_silver.limit(5).toPandas()

In [0]:
#df_silver.write\
 #   .format("delta")\
 ##  .saveAsTable("projeto_profissional_bigdata.datatran_2018.silver_datatran18")

In [0]:
#df = spark.table("projeto_profissional_bigdata.datatran_2018.silver_datatran18")

#df.limit(20).toPandas()

In [0]:
#%sql
#select * from projeto_profissional_bigdata.datatran_2018.silver_datatran18 limit (20);